# Week 1 Day 2 Challenge Solution

This notebook upgrades the Day 1 webpage summarizer to run with:
- **Ollama** locally (default)
- **Gemini** via the OpenAI-compatible endpoint (API key from `.env`)

Set `provider="ollama"` or `provider="gemini"` when calling `display_summary(...)`.

In [1]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
# Load env vars
load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

OLLAMA_BASE_URL = "http://localhost:11434/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

DEFAULT_MODELS = {
    "ollama": "llama3.2",
    "gemini": "gemini-2.5-flash-lite",
}

print("Environment loaded.")
print(f"GOOGLE_API_KEY configured: {bool(GOOGLE_API_KEY)}")

Environment loaded.
GOOGLE_API_KEY configured: True


In [3]:
def ensure_ollama_running() -> None:
    try:
        response = requests.get("http://localhost:11434", timeout=3)
        print(response.text)
    except requests.RequestException as exc:
        raise RuntimeError(
            "Ollama is not reachable at http://localhost:11434. Start it with `ollama serve`."
        ) from exc


def get_client(provider: str) -> OpenAI:
    provider = provider.lower()

    if provider == "ollama":
        ensure_ollama_running()
        return OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

    if provider == "gemini":
        if not GOOGLE_API_KEY:
            raise RuntimeError("Missing GOOGLE_API_KEY in .env")
        return OpenAI(base_url=GEMINI_BASE_URL, api_key=GOOGLE_API_KEY)

    raise ValueError("provider must be 'ollama' or 'gemini'")

In [4]:
def fetch_website_contents(url: str, max_chars: int = 15000) -> str:
    """Fetch and clean webpage text for summarization."""
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; llm-engineering-bot/1.0)"},
        timeout=20,
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "form", "svg"]):
        tag.decompose()

    title = soup.title.get_text(strip=True) if soup.title else "Untitled Page"
    body_text = "\n".join(line.strip() for line in soup.get_text("\n").splitlines() if line.strip())
    body_text = body_text[:max_chars]

    return f"URL: {url}\nTITLE: {title}\n\nCONTENT:\n{body_text}"

In [5]:
SYSTEM_PROMPT = """
You are a concise assistant that summarizes webpage content.
Ignore obvious navigation/boilerplate text.
Return markdown only (no code fences).
"""

USER_PROMPT_PREFIX = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, summarize those too.

"""


def summarize(url: str, provider: str = "ollama", model: str | None = None) -> str:
    provider = provider.lower()
    model = model or DEFAULT_MODELS[provider]

    website_text = fetch_website_contents(url)
    client = get_client(provider)

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT_PREFIX + website_text},
        ],
        temperature=0.2,
    )

    return response.choices[0].message.content


def display_summary(url: str, provider: str = "ollama", model: str | None = None) -> None:
    summary = summarize(url=url, provider=provider, model=model)
    display(Markdown(summary))

In [7]:
# Quick checks
# 1) Ollama health
ensure_ollama_running()

# 2) Test local model (pull first if needed: `ollama pull llama3.2`)
display_summary("https://edwarddonner.com", provider="ollama")

Ollama is running
Ollama is running


**Summary**: The website is the personal blog of Edward Donner, a tech enthusiast and entrepreneur. He shares his interests in AI, coding, music production, and self-improvement. As the co-founder and CTO of Nebula.io, he discusses the application of AI to help people discover their potential.

**News/Announcements**: The website includes several recent announcements:

* February 17, 2026: "AI Coder: Vibe Coder to Agentic Engineer"
* January 4, 2026: "AI Builder with n8n – Create Agents and Voice Agents"
* November 11, 2025: "The Unique Energy of an AI Live Event"
* September 15, 2025: "AI Engineering MLOps Track – Deploy AI to Production"

In [8]:
# Optional Gemini test (requires GOOGLE_API_KEY in .env)
if GOOGLE_API_KEY:
    display_summary("https://edwarddonner.com", provider="gemini")
else:
    print("Skipping Gemini test: GOOGLE_API_KEY not found in .env")

Edward Donner is a co-founder and CTO of Nebula.io, an AI company focused on helping people discover their potential. He is also the founder of untapt, an AI startup acquired in 2021. Donner is passionate about writing code and experimenting with LLMs, and has created best-selling Udemy courses on the topic.

Recent posts include:
*   AI Coder: Vibe Coder to Agentic Engineer (February 17, 2026)
*   AI Builder with n8n – Create Agents and Voice Agents (January 4, 2026)
*   The Unique Energy of an AI Live Event (November 11, 2025)
*   AI Engineering MLOps Track – Deploy AI to Production (September 15, 2025)